In [1]:
# BLOCK A: 全域設定 (CONFIG) - 所有模組開關都在這裡控制
# ============================================================
import os
import re
import ast
import random
import copy
from collections import defaultdict, Counter
 
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image, ImageDraw
import timm
import time
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix
 
CONFIG = {
    # ---- 路徑設定 ----
    "META_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/ptbxl_database.csv",
    "SCP_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/scp_statements.csv",
    "IMG_ROOT": "/kaggle/input/datasets/bjoernjostein/ptb-xl-ecg-image-gmc2024",
    "IMG_RATE": "lr",          # 'lr' (100Hz) 或 'hr' (500Hz)，需與影像資料集實際命名一致
    "MAX_IMG_PER_RECORD": 1,   # 每筆 ecg_id 取幾張合成影像 (1 = 只取 -0，避免資料洩漏疑慮)
    "OUTPUT_DIR": "/kaggle/working/outputs",
 
    # ---- 任務設定 ----
    "SINGLE_LABEL_ONLY": True,     # True: 只保留單一 superclass 的紀錄 (簡化成單標籤分類)
    "AGE_CLIP_MAX": 90,            # PTB-XL 對 >89 歲設為 300，需 clip
 
    # ---- 多模態開關 ----
    "USE_AGE_MODALITY": True,      # 關掉即退化為純影像分類 (方便做 ablation 比較)
    "USE_SEX_MODALITY": True,      # 性別開關，可與年齡獨立控制
    "DEMO_EMBED_DIM": 16,          # age+sex 合併後的人口學特徵 embedding 維度
 
    # ---- 模型設定 ----
    "BACKBONE": "convnext_atto",    #  convnext_atto   (3.7M)  convnext_femto  (5M)   convnext_tiny  (28M)
                                    #   resnet18        (11M)   resnet50         (25M)
                                    #   efficientnet_b0 (5M)    efficientnet_b2  (9M)
    "DROP_PATH_RATE": 0.1,          # ConvNeXt/EfficientNet/ViT系列的Stochastic Depth正則化，ResNet等不支援的架構會自動忽略此設定
    "PRETRAINED": True,
    "NUM_CLASSES": 5,               # NORM / MI / STTC / CD / HYP
    "IMG_SIZE": 224,
    "MASK_TOP_RATIO": 0.3,          # 遮蔽影像上方比例，避免學到病人編號等識別資訊
 
    # ---- 資料增強設定 (Data Augmentation) ----
    "AUG_ROTATION_DEGREE": 5,          # 隨機旋轉角度 (+- 5度)
    "AUG_TRANSLATE": (0.05, 0.05),     # 隨機平移 (寬度平移5%, 高度平移5%)
    "AUG_SCALE": (0.9, 1.1),           # 隨機縮放 (90% ~ 110%)
    "AUG_SHEAR": 3,                    # 隨機剪切角度
    "AUG_COLOR_JITTER": True,          # 亮度與對比度微調開關
    "AUG_BRIGHTNESS": 0.15,
    "AUG_CONTRAST": 0.15,
    "AUG_RANDOM_ERASING_P": 0.2,       # 隨機擦除機率
 
    # ---- 訓練設定 ----
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 2,
    "EPOCHS": 30,                  # 拉長訓練，配合降低的LR讓模型充分收斂
    "FREEZE_EPOCHS": 5,            # 前 N epoch 凍結 image backbone，只訓練 fusion head
    "LR_HEAD": 5e-4,
    "LR_BACKBONE": 3e-5,           # 降低，避免backbone解凍瞬間暴衝
    "WEIGHT_DECAY": 1e-3,           # 1e-4 -> 1e-3，加重正則化力道，抑制過擬合
    "SEED": 42,
 
    # ---- Scheduler 選擇 ----
    "SCHEDULER_TYPE": "plateau",   # "plateau" (ReduceLROnPlateau, 表現驅動) 或 "cosine" (CosineAnnealingLR, 時間驅動)
 
    # ---- 不平衡處理開關 ----
    "USE_WEIGHTED_SAMPLER": True,
    "USE_CLASS_WEIGHTED_LOSS": True,
    "USE_FOCAL_LOSS": False,       # 與 USE_CLASS_WEIGHTED_LOSS 二選一，Focal 優先
    "FOCAL_GAMMA": 2.0,
 
    # ---- 資料增強開關 ----
    "USE_MIXUP": False,            # 對 ECG 印刷影像做 Mixup 需謹慎，預設關閉
    "MIXUP_ALPHA": 0.2,
 
    # ---- Grad-CAM 開關 ----
    "USE_GRADCAM": True,
    "GRADCAM_TARGET_LAYER": "stages.3",  # 依 backbone 調整 (resnet 系列用 layer4 ConvNeXt 用 'stages.3')
 
    # ---- Early stopping ----
    "EARLY_STOP_METRIC": "val_f1_macro",  # 用 F1 而非 accuracy/val_loss 做早停判準
    "EARLY_STOP_PATIENCE": 8,      # 15 -> 8，val_loss明顯惡化後不需撐太久

    # ---- 訓練Log顯示功能 ----
    "VERBOSE_BATCH": True,   # True才會印 batch-level進度，正式訓練建議關閉保持log簡潔
}
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
 
print('A')

A


In [2]:
# BLOCK B: 隨機種子固定 (可重現性)
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["SEED"])

print('B')

B


In [3]:
# BLOCK C: PTB-XL Metadata 讀取與 diagnostic superclass 聚合
# ============================================================
def load_ptbxl_metadata(meta_csv: str, scp_csv: str, single_label_only: bool = True) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv, index_col="ecg_id")
    meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)

    agg_df = pd.read_csv(scp_csv, index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1]

    def aggregate_diagnostic(scp_codes: dict):
        classes = set()
        for code in scp_codes.keys():
            if code in agg_df.index:
                classes.add(agg_df.loc[code].diagnostic_class)
        return list(classes)

    meta["diagnostic_superclass"] = meta["scp_codes"].apply(aggregate_diagnostic)
    meta = meta[meta["diagnostic_superclass"].apply(len) > 0].copy()

    if single_label_only:
        meta = meta[meta["diagnostic_superclass"].apply(len) == 1].copy()
        meta["label"] = meta["diagnostic_superclass"].apply(lambda x: x[0])
    else:
        # 多標籤模式：保留 list，訓練時需搭配 BCEWithLogitsLoss (本檔案預設走單標籤路徑)
        meta["label"] = meta["diagnostic_superclass"]

    return meta

print('C')

C


In [4]:
# BLOCK D: 年齡清理 + 影像路徑展開 (含資料夾分層規則)
# ============================================================
def get_ptbxl_folder(ecg_id: int) -> str:
    """ecg_id -> 千位分組資料夾名稱，例如 12345 -> '12000'"""
    return f"{(ecg_id // 1000) * 1000:05d}"


def clean_age(meta: pd.DataFrame, clip_max: int) -> pd.DataFrame:
    meta = meta.copy()

    n_before = len(meta)
    meta = meta[meta["age"].notna()].copy()
    n_dropped = n_before - len(meta)
    if n_dropped > 0:
        print(f"[clean_age] 移除 {n_dropped} 筆缺少年齡的紀錄 ({n_before} -> {len(meta)})")

    # PTB-XL 對 >89 歲的紀錄設為 300 歲 (HIPAA 去識別化)，需特別處理
    meta["is_elderly_capped"] = meta["age"] >= 200
    meta["age_clean"] = meta["age"].clip(upper=clip_max)
    return meta

def clean_sex(meta: pd.DataFrame) -> pd.DataFrame:
    """PTB-XL sex 欄位: 0=男性, 1=女性"""
    meta = meta.copy()
    n_before = len(meta)
    meta = meta[meta["sex"].notna()].copy()
    n_dropped = n_before - len(meta)
    if n_dropped > 0:
        print(f"[clean_sex] 移除 {n_dropped} 筆缺少性別的紀錄 ({n_before} -> {len(meta)})")
    meta["sex_clean"] = meta["sex"].astype(int)
    return meta


def build_image_index(meta_df: pd.DataFrame, img_root: str, rate: str,
                       max_img_per_record: int) -> pd.DataFrame:
    rows = []
    for ecg_id, row in meta_df.iterrows():
        folder = get_ptbxl_folder(ecg_id)
        for idx in range(max_img_per_record):
            fname = f"{ecg_id:05d}_{rate}-{idx}.png"
            fpath = os.path.join(img_root, folder, fname)
            if os.path.exists(fpath):
                rows.append({
                    "img_path": fpath,
                    "ecg_id": ecg_id,
                    "patient_id": row["patient_id"],
                    "age_clean": row["age_clean"],
                    "sex_clean": row["sex_clean"],
                    "label": row["label"],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            "沒有找到任何符合命名規則的影像檔案，請先確認 IMG_ROOT / IMG_RATE / 檔名格式是否正確。"
        )
    return df

print('D')

D


In [5]:
# BLOCK E: Patient-level train/val/test 切分 (避免資料洩漏)
# ============================================================
def patient_level_split(df: pd.DataFrame, seed: int, val_size=0.15, test_size=0.15):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(df, groups=df["patient_id"]))
    trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    relative_val_size = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval_df, groups=trainval_df["patient_id"]))
    train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
    val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

    # 檢查 patient_id 不重疊
    assert set(train_df.patient_id) & set(val_df.patient_id) == set()
    assert set(train_df.patient_id) & set(test_df.patient_id) == set()
    assert set(val_df.patient_id) & set(test_df.patient_id) == set()

    return train_df, val_df, test_df


print('E')

E


In [6]:
# BLOCK F: 年齡標準化 (用 train set 統計量，避免資訊洩漏)
# ============================================================
class AgeScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, ages: np.ndarray):
        self.mean = float(np.mean(ages))
        self.std = float(np.std(ages) + 1e-6)
        return self

    def transform(self, ages: np.ndarray):
        return (ages - self.mean) / self.std

print('F')

F


In [7]:
# BLOCK G: 影像資料增強 (Augmentation)
# ============================================================
class MaskTopRegion:
    """
    遮蔽影像最上方一定比例的區域，避免模型學到病人編號、日期等列印在
    ECG影像頂端的識別資訊（防止資料洩漏/走捷徑學習不具臨床意義的特徵）。
    """
    def __init__(self, top_ratio: float = 0.2, fill_color=(255, 255, 255)):
        self.top_ratio = top_ratio
        self.fill_color = fill_color

    def __call__(self, img):
        img = img.copy()
        w, h = img.size
        mask_height = int(h * self.top_ratio)
        draw = ImageDraw.Draw(img)
        draw.rectangle([0, 0, w, mask_height], fill=self.fill_color)
        return img


def get_transforms(cfg: dict):
    img_size = cfg["IMG_SIZE"]
    mask_top_ratio = cfg["MASK_TOP_RATIO"]
    mask_top = MaskTopRegion(top_ratio=mask_top_ratio, fill_color=(255, 255, 255))

    # 組裝 Train Pipeline
    train_tf_list = [
        mask_top,
        transforms.Resize((img_size, img_size)),
        # 使用 RandomAffine 一次涵蓋 旋轉、平移、縮放(放大縮小)、剪切
        transforms.RandomAffine(
            degrees=cfg.get("AUG_ROTATION_DEGREE", 5),
            translate=cfg.get("AUG_TRANSLATE", (0.05, 0.05)),
            scale=cfg.get("AUG_SCALE", (0.9, 1.1)),
            shear=cfg.get("AUG_SHEAR", 3),
            fill=255 # 變換邊界填充白色（符合心電圖背景）
        ),
    ]

    # 色彩微調開關
    if cfg.get("AUG_COLOR_JITTER", False):
        train_tf_list.append(
            transforms.ColorJitter(
                brightness=cfg.get("AUG_BRIGHTNESS", 0.15),
                contrast=cfg.get("AUG_CONTRAST", 0.15)
            )
        )

    # Tensor 轉換與正規化
    train_tf_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # 隨機擦除 (必須在 ToTensor 之後)
    if cfg.get("AUG_RANDOM_ERASING_P", 0.0) > 0:
        train_tf_list.append(
            transforms.RandomErasing(
                p=cfg["AUG_RANDOM_ERASING_P"],
                scale=(0.02, 0.08),
                value=1.0 # 擦除區域填滿白色
            )
        )

    train_tf = transforms.Compose(train_tf_list)

    # Validation/Testing 保持乾淨，只做必要的預處理
    eval_tf = transforms.Compose([
        mask_top,
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    return train_tf, eval_tf
def visualize_augmentations(image_path: str, cfg: dict, n_examples: int = 5, output_dir: str = None):
    """
    從指定影像產生: 原圖 / 遮罩後 / N張隨機增強範例，並列顯示。
    用於訓練前確認資料增強的強度與效果符合預期，不用等訓練跑完才發現設定不對。
    """
    import matplotlib.pyplot as plt

    def _denorm(img_tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
        img = img_tensor.clone().cpu().numpy().transpose(1, 2, 0)
        img = img * np.array(std) + np.array(mean)
        return np.clip(img, 0, 1)

    train_tf, eval_tf = get_transforms(cfg)
    raw_img = Image.open(image_path).convert("RGB")

    mask_top = MaskTopRegion(top_ratio=cfg["MASK_TOP_RATIO"], fill_color=(255, 255, 255))
    masked_preview = mask_top(raw_img).resize((cfg["IMG_SIZE"], cfg["IMG_SIZE"]))

    n_cols = n_examples + 2
    fig, axes = plt.subplots(1, n_cols, figsize=(3 * n_cols, 3.4))

    axes[0].imshow(raw_img)
    axes[0].set_title("原圖\n(Original)", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(masked_preview)
    axes[1].set_title(f"遮罩後\n(top {int(cfg['MASK_TOP_RATIO']*100)}%)", fontsize=10)
    axes[1].axis("off")

    for i in range(n_examples):
        aug_tensor = train_tf(raw_img)
        aug_img = _denorm(aug_tensor)
        axes[2 + i].imshow(aug_img)
        axes[2 + i].set_title(f"增強範例 #{i+1}", fontsize=10)
        axes[2 + i].axis("off")

    plt.suptitle("資料增強範例 (Data Augmentation Preview)", fontsize=13, fontweight="bold")
    plt.tight_layout()

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        path = os.path.join(output_dir, "augmentation_preview.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"[preview] 儲存: {path}")

    plt.show()
    plt.close()
print('G')

G


In [8]:
# BLOCK H: 多模態 Dataset
# ============================================================
class PTBXLMultimodalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, age_scaler: AgeScaler,
                 label2idx: dict, use_age: bool = True, use_sex: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.age_scaler = age_scaler
        self.label2idx = label2idx
        self.use_age = use_age
        self.use_sex = use_sex

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["img_path"]).convert("RGB")
        image = self.transform(image)

        demo_values = []
        if self.use_age:
            age_norm = self.age_scaler.transform(np.array([row["age_clean"]]))[0]
            demo_values.append(age_norm)
        if self.use_sex:
            demo_values.append(float(row["sex_clean"]))

        if len(demo_values) > 0:
            demo_tensor = torch.tensor(demo_values, dtype=torch.float32)
        else:
            demo_tensor = torch.tensor([0.0], dtype=torch.float32)

        label = self.label2idx[row["label"]]
        return image, demo_tensor, label

print('H')



H


In [9]:
# BLOCK I: WeightedRandomSampler 建構 (處理類別不平衡)
# ============================================================
def build_weighted_sampler(labels: list) -> WeightedRandomSampler:
    class_counts = Counter(labels)
    num_samples = len(labels)
    class_weights = {c: num_samples / count for c, count in class_counts.items()}
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(
        weights=sample_weights, num_samples=num_samples, replacement=True
    )
print('I')

I


In [10]:
# BLOCK J: DataLoader 建構
# ============================================================
def build_dataloaders(train_df, val_df, test_df, label2idx, cfg: dict):
    train_tf, eval_tf = get_transforms(cfg)
 
    age_scaler = AgeScaler().fit(train_df["age_clean"].values)
 
    train_ds = PTBXLMultimodalDataset(train_df, train_tf, age_scaler, label2idx,
                                       cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
    val_ds = PTBXLMultimodalDataset(val_df, eval_tf, age_scaler, label2idx,
                                     cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
    test_ds = PTBXLMultimodalDataset(test_df, eval_tf, age_scaler, label2idx,
                                      cfg["USE_AGE_MODALITY"], cfg["USE_SEX_MODALITY"])
 
    if cfg["USE_WEIGHTED_SAMPLER"]:
        train_labels_idx = [label2idx[l] for l in train_df["label"]]
        sampler = build_weighted_sampler(train_labels_idx)
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler,
                                   num_workers=cfg["NUM_WORKERS"])
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], shuffle=True,
                                   num_workers=cfg["NUM_WORKERS"])
 
    val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                             num_workers=cfg["NUM_WORKERS"])
    test_loader = DataLoader(test_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                              num_workers=cfg["NUM_WORKERS"])
 
    return train_loader, val_loader, test_loader, age_scaler
print('J')

J


In [11]:
# BLOCK K: 多模態模型 (Image Encoder + Age Encoder + Fusion Head)
# ============================================================
class MultimodalECGNet(nn.Module):
    def __init__(self, backbone: str, pretrained: bool, num_classes: int,
                 demo_embed_dim: int = 16, use_age: bool = True, use_sex: bool = True):
        super().__init__()
        self.use_age = use_age
        self.use_sex = use_sex
        self.use_demo = use_age or use_sex

        self.img_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        img_feat_dim = self.img_encoder.num_features

        demo_input_dim = int(use_age) + int(use_sex)

        if self.use_demo:
            self.demo_encoder = nn.Sequential(
                nn.Linear(demo_input_dim, demo_embed_dim),
                nn.ReLU(),
                nn.Linear(demo_embed_dim, demo_embed_dim),
                nn.ReLU(),
            )
            fusion_in_dim = img_feat_dim + demo_embed_dim
        else:
            self.demo_encoder = None
            fusion_in_dim = img_feat_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, image, demo):
        img_feat = self.img_encoder(image)
        if self.use_demo:
            demo_feat = self.demo_encoder(demo)
            fused = torch.cat([img_feat, demo_feat], dim=1)
        else:
            fused = img_feat
        return self.classifier(fused)

    def freeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = True
class MultimodalECGNet(nn.Module):
    def __init__(self, backbone: str, pretrained: bool, num_classes: int,
                 demo_embed_dim: int = 16, use_age: bool = True, use_sex: bool = True,
                 drop_path_rate: float = 0.0):
        super().__init__()
        self.use_age = use_age
        self.use_sex = use_sex
        self.use_demo = use_age or use_sex

        self.img_encoder = self._build_img_encoder(backbone, pretrained, drop_path_rate)
        img_feat_dim = self.img_encoder.num_features

        demo_input_dim = int(use_age) + int(use_sex)

        if self.use_demo:
            self.demo_encoder = nn.Sequential(
                nn.Linear(demo_input_dim, demo_embed_dim),
                nn.ReLU(),
                nn.Linear(demo_embed_dim, demo_embed_dim),
                nn.ReLU(),
            )
            fusion_in_dim = img_feat_dim + demo_embed_dim
        else:
            self.demo_encoder = None
            fusion_in_dim = img_feat_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    @staticmethod
    def _build_img_encoder(backbone: str, pretrained: bool, drop_path_rate: float):
        """
        建立影像backbone。drop_path_rate (Stochastic Depth) 只有ConvNeXt/EfficientNet/ViT
        等含大量堆疊殘差區塊的架構支援，ResNet系列傳入這個參數會直接TypeError，
        所以用try/except自動偵測，不支援時忽略此設定並印出提醒，不需要每次換backbone
        手動判斷要不要傳這個參數。
        """
        if drop_path_rate and drop_path_rate > 0:
            try:
                return timm.create_model(backbone, pretrained=pretrained, num_classes=0,
                                          drop_path_rate=drop_path_rate)
            except TypeError:
                print(f"[warn] backbone='{backbone}' 不支援 drop_path_rate 參數，已忽略此設定，"
                      f"改用標準建構方式（此為預期行為，常見於ResNet系列）")
        return timm.create_model(backbone, pretrained=pretrained, num_classes=0)

    def forward(self, image, demo):
        img_feat = self.img_encoder(image)
        if self.use_demo:
            demo_feat = self.demo_encoder(demo)
            fused = torch.cat([img_feat, demo_feat], dim=1)
        else:
            fused = img_feat
        return self.classifier(fused)

    def freeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = True

print('K')

K


In [12]:
# BLOCK L: Loss function (class-weighted CrossEntropy 或 Focal Loss)
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def build_criterion(train_labels_idx: list, num_classes: int, cfg: dict):
    counts = Counter(train_labels_idx)
    weights = torch.tensor(
        [len(train_labels_idx) / counts.get(c, 1) for c in range(num_classes)],
        dtype=torch.float32,
    ).to(DEVICE)
    # 常見做法：weight clipping，避免極端不平衡讓多數類別 recall 崩掉
    weights = torch.clamp(weights, min=0.5, max=5.0)

    if cfg["USE_FOCAL_LOSS"]:
        return FocalLoss(alpha=weights, gamma=cfg["FOCAL_GAMMA"])
    elif cfg["USE_CLASS_WEIGHTED_LOSS"]:
        return nn.CrossEntropyLoss(weight=weights)
    else:
        return nn.CrossEntropyLoss()

print('L')

L


In [13]:
# BLOCK M: Optimizer + Freeze/Unfreeze 排程
# ============================================================
def build_optimizer(model: MultimodalECGNet, cfg: dict):
    backbone_params = list(model.img_encoder.parameters())
    other_params = list(model.classifier.parameters())
    if model.use_demo:
        other_params += list(model.demo_encoder.parameters())

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["LR_BACKBONE"]},
        {"params": other_params, "lr": cfg["LR_HEAD"]},
    ], weight_decay=cfg["WEIGHT_DECAY"])
    return optimizer
print('M')

M


In [14]:
# BLOCK N: LR Scheduler
# ============================================================
def build_scheduler(optimizer, cfg: dict):
    if cfg["SCHEDULER_TYPE"] == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])
    else:  # "plateau" (預設)
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,           # 注意: 要小於EARLY_STOP_PATIENCE，否則同時觸發
            min_lr=1e-6,
        )


def step_scheduler(scheduler, cfg: dict, metric_value: float):
    """統一入口：依 SCHEDULER_TYPE 決定 step() 要不要傳指標值。
    Block R 只需呼叫這個函式，之後在 CONFIG 切換 scheduler 種類都不用改 Block R。"""
    if cfg["SCHEDULER_TYPE"] == "cosine":
        scheduler.step()
    else:
        scheduler.step(metric_value)

print('N')

N


In [15]:
# BLOCK O: Mixup (只作用於影像分支，年齡分支維持原值傳遞，預設關閉)
# ============================================================
def mixup_data(images, ages, labels, alpha: float):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = images.size(0)
    index = torch.randperm(batch_size).to(images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    # 年齡不做混合，保留原始年齡對應原始影像的邏輯較合理；若要混合可改成加權平均
    labels_a, labels_b = labels, labels[index]
    return mixed_images, ages, labels_a, labels_b, lam


def mixup_criterion(criterion, pred, labels_a, labels_b, lam):
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


print('O')

O


In [16]:
# BLOCK P: 訓練 / 驗證迴圈
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, cfg: dict):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
 
    for batch_idx, (images, demo, labels) in enumerate(loader):
        images, demo, labels = images.to(DEVICE), demo.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
 
        if cfg["USE_MIXUP"]:
            mixed_images, demo, labels_a, labels_b, lam = mixup_data(images, demo, labels, cfg["MIXUP_ALPHA"])
            outputs = model(mixed_images, demo)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images, demo)
            loss = criterion(outputs, labels)
 
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
 
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += images.size(0)
 
        if cfg.get("VERBOSE_BATCH", False) and batch_idx % 50 == 0:
            print(f"  batch {batch_idx}/{len(loader)} loss={loss.item():.4f}")
 
    avg_loss = total_loss / len(loader.dataset)
    accuracy = total_correct / total_samples
    return avg_loss, accuracy
 
 
@torch.no_grad()
def evaluate(model, loader, criterion, idx2label: dict):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    for images, demo, labels in loader:
        images, demo, labels = images.to(DEVICE), demo.to(DEVICE), labels.to(DEVICE)
        outputs = model(images, demo)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
 
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
 
    val_loss = total_loss / len(loader.dataset)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
 
    from sklearn.metrics import precision_score, recall_score, accuracy_score
    precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    accuracy = accuracy_score(all_labels, all_preds)
 
    report = classification_report(
        all_labels, all_preds,
        target_names=[idx2label[i] for i in range(len(idx2label))],
        zero_division=0,
    )
    cm = confusion_matrix(all_labels, all_preds)
 
    return {
        "val_loss": val_loss,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "val_accuracy": accuracy,
        "report": report, "cm": cm,
        "all_labels": all_labels, "all_preds": all_preds,
    }
print('P')

P


In [17]:
# BLOCK Q: Grad-CAM (只作用於影像分支)
# ============================================================
class GradCAM:
    def __init__(self, model: MultimodalECGNet, target_layer_name: str):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer = dict([*model.img_encoder.named_modules()])[target_layer_name]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, demo_tensor, class_idx=None):
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
        demo_tensor = demo_tensor.unsqueeze(0).to(DEVICE)

        output = self.model(image_tensor, demo_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    # 注意: Grad-CAM 只反映影像分支的空間注意力，年齡/性別分支的貢獻無法用此方式視覺化，
    # 解讀結果時務必註明「此為影像模態的注意力，不代表人口學模態的影響程度」。


def denormalize_image(img_tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """把經過 ImageNet Normalize 的 tensor 還原成可顯示的 [0,1] RGB 影像"""
    img = img_tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)


def plot_gradcam_single(image_tensor, cam, pred_label: str, true_label: str, save_path: str):
    """單筆樣本的 Grad-CAM 三合一圖: 原圖 / 熱力圖 / 疊圖"""
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm

    img = denormalize_image(image_tensor)
    heatmap_rgb = cm.jet(cam)[..., :3]
    overlay = np.clip(0.55 * img + 0.45 * heatmap_rgb, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original ECG Image")
    axes[0].axis("off")

    axes[1].imshow(cam, cmap="jet")
    axes[1].set_title("Grad-CAM Heatmap\n(模型重點關注區域)")
    axes[1].axis("off")

    correct = "✓" if pred_label == true_label else "✗"
    axes[2].imshow(overlay)
    axes[2].set_title(f"Overlay  [{correct}]  pred={pred_label} / true={true_label}")
    axes[2].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()


def generate_gradcam_examples(model, test_loader, idx2label: dict, target_layer: str, output_dir: str,
                               n_per_class: int = 1):
    """每個類別各挑 n_per_class 筆 test 樣本，產出 Grad-CAM 視覺化，方便逐類別檢視模型關注區域"""
    gradcam = GradCAM(model, target_layer)
    dataset = test_loader.dataset

    class_to_indices = {i: [] for i in range(len(idx2label))}
    for idx in range(len(dataset)):
        label = dataset.df.iloc[idx]["label"]
        label_idx = [k for k, v in idx2label.items() if v == label][0]
        class_to_indices[label_idx].append(idx)

    saved_paths = []
    for class_idx, indices in class_to_indices.items():
        class_name = idx2label[class_idx]
        chosen = indices[:n_per_class]
        for i, sample_idx in enumerate(chosen):
            image_tensor, demo_tensor, true_idx = dataset[sample_idx]
            cam, pred_idx = gradcam.generate(image_tensor, demo_tensor)
            save_path = os.path.join(output_dir, f"gradcam_{class_name}_{i}.png")
            plot_gradcam_single(image_tensor, cam, idx2label[pred_idx], idx2label[true_idx], save_path)
            saved_paths.append(save_path)
            print(f"[gradcam] 儲存: {save_path}")

    return saved_paths
print('Q')

Q


In [18]:
# BLOCK R: 主流程 (串接所有模組)
# ============================================================
def main(cfg: dict):
    print(f"Device: {DEVICE}")
 
    # --- 資料準備 ---
    meta = load_ptbxl_metadata(cfg["META_CSV"], cfg["SCP_CSV"], cfg["SINGLE_LABEL_ONLY"])
    meta = clean_age(meta, cfg["AGE_CLIP_MAX"])
    meta = clean_sex(meta)
    long_df = build_image_index(meta, cfg["IMG_ROOT"], cfg["IMG_RATE"], cfg["MAX_IMG_PER_RECORD"])
    print(f"總影像數: {len(long_df)}, 類別分布:\n{long_df['label'].value_counts()}")
 
    labels_sorted = sorted(long_df["label"].unique())
    label2idx = {l: i for i, l in enumerate(labels_sorted)}
    idx2label = {i: l for l, i in label2idx.items()}
 
    train_df, val_df, test_df = patient_level_split(long_df, cfg["SEED"])
    print(f"train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}")
 
    train_loader, val_loader, test_loader, age_scaler = build_dataloaders(
        train_df, val_df, test_df, label2idx, cfg
    )
 
    # --- 模型 / loss / optimizer ---
    model = MultimodalECGNet(
        backbone=cfg["BACKBONE"], pretrained=cfg["PRETRAINED"],
        num_classes=cfg["NUM_CLASSES"], demo_embed_dim=cfg["DEMO_EMBED_DIM"],
        use_age=cfg["USE_AGE_MODALITY"], use_sex=cfg["USE_SEX_MODALITY"],
        drop_path_rate=cfg.get("DROP_PATH_RATE", 0.0),   # 新增這行
    ).to(DEVICE)
 
    train_labels_idx = [label2idx[l] for l in train_df["label"]]
    criterion = build_criterion(train_labels_idx, cfg["NUM_CLASSES"], cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)
 
    # --- 訓練迴圈 (含 freeze/unfreeze + early stopping) ---
    best_metric = -np.inf
    best_state = None
    best_epoch = None
    patience_counter = 0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_accuracy": [],
               "val_f1_macro": [], "val_precision_macro": [], "val_recall_macro": [], "val_accuracy": []}
 
    def _fmt_lr(x):
        s = f"{x:.1e}"
        mantissa, exp = s.split("e")
        mantissa = mantissa.rstrip("0").rstrip(".")
        exp = exp.replace("+0", "+").replace("-0", "-").replace("+", "")
        return f"{mantissa}e{exp}"
 
    backbone_name = cfg["BACKBONE"]
    print(f"\n開始訓練 {backbone_name}")
    print("=" * 55)
    if cfg["FREEZE_EPOCHS"] > 0:
        print("  Backbone 已凍結，僅訓練分類頭")
 
    for epoch in range(cfg["EPOCHS"]):
        epoch_start = time.time()
 
        if epoch < cfg["FREEZE_EPOCHS"]:
            model.freeze_backbone()
        else:
            if epoch == cfg["FREEZE_EPOCHS"]:
                print(f"\n🔥 [{backbone_name}] Epoch {epoch+1}：解凍 backbone")
                print(f"  分層學習率：分類頭 {_fmt_lr(cfg['LR_HEAD'])} / backbone {_fmt_lr(cfg['LR_BACKBONE'])}")
            model.unfreeze_backbone()
 
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, cfg)
        val_metrics = evaluate(model, val_loader, criterion, idx2label)
        step_scheduler(scheduler, cfg, val_metrics["val_f1_macro"])
 
        epoch_time = time.time() - epoch_start
 
        print(f"[{backbone_name}] Epoch {epoch+1:02d}/{cfg['EPOCHS']} | "
              f"Loss: {train_loss:.4f}/{val_metrics['val_loss']:.4f} | "
              f"Acc: {train_acc:.4f}/{val_metrics['val_accuracy']:.4f} | "
              f"Prec: {val_metrics['val_precision_macro']:.4f} | "
              f"Rec: {val_metrics['val_recall_macro']:.4f} | "
              f"F1: {val_metrics['val_f1_macro']:.4f} | "
              f"Time: {epoch_time:.1f}s")
 
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_f1_macro"].append(val_metrics["val_f1_macro"])
        history["val_precision_macro"].append(val_metrics["val_precision_macro"])
        history["val_recall_macro"].append(val_metrics["val_recall_macro"])
        history["val_accuracy"].append(val_metrics["val_accuracy"])
 
        current_metric = val_metrics[cfg["EARLY_STOP_METRIC"]]
        if current_metric > best_metric:
            best_metric = current_metric
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience_counter = 0
            print(f"  → ⭐ 已儲存最佳模型（F1: {val_metrics['val_f1_macro']:.4f}）")
        else:
            patience_counter += 1
            print(f"  → 未進步（{patience_counter}/{cfg['EARLY_STOP_PATIENCE']}）")
            if patience_counter >= cfg["EARLY_STOP_PATIENCE"]:
                print(f"\nEarly stopping at epoch {epoch+1} (best {cfg['EARLY_STOP_METRIC']}={best_metric:.4f})")
                break
 
    # --- 載回最佳權重，跑 test set ---
    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, criterion, idx2label)
    print("\n=== Test Set 結果 ===")
    print(test_metrics["report"])
    print("Confusion Matrix:\n", test_metrics["cm"])
    print()
    print_overall_summary(cfg["BACKBONE"], test_metrics)
 
    torch.save(model.state_dict(), os.path.join(cfg["OUTPUT_DIR"], "best_model.pth"))
 
    # --- 自動產出所有圖表 ---
    class_names = [idx2label[i] for i in range(len(idx2label))]
    generate_all_plots(
        history=history,
        freeze_epochs=cfg["FREEZE_EPOCHS"],
        best_epoch=best_epoch,
        test_metrics=test_metrics,
        all_labels=test_metrics["all_labels"],
        all_preds=test_metrics["all_preds"],
        class_names=class_names,
        output_dir=cfg["OUTPUT_DIR"],
    )
 
    # --- Grad-CAM 視覺化 (每個類別各挑1筆test樣本，產出原圖/熱力圖/疊圖) ---
    if cfg["USE_GRADCAM"]:
        generate_gradcam_examples(
            model=model,
            test_loader=test_loader,
            idx2label=idx2label,
            target_layer=cfg["GRADCAM_TARGET_LAYER"],
            output_dir=cfg["OUTPUT_DIR"],
            n_per_class=1,
        )
 
    return model, test_metrics
print('R')

R


In [19]:
# BLOCK S: 視覺化 (訓練曲線 / 混淆矩陣 / 各類別指標)
# ============================================================
def setup_cjk_font():
    """設定中文字型，避免 matplotlib CJK 缺字警告。找不到就靜默回退英文。"""
    import matplotlib
    from matplotlib import font_manager
    candidates = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    ]
    for path in candidates:
        if os.path.exists(path):
            font_manager.fontManager.addfont(path)
            name = font_manager.FontProperties(fname=path).get_name()
            matplotlib.rcParams["font.family"] = name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return True
    return False
 
 
def plot_training_curves(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train Loss", color="#E2B08C")
    axes[0].plot(epochs, history["val_loss"], marker="s", label="Val Loss", color="#8C9EE2")
    axes[0].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6,
                     label=f"Backbone unfreeze (epoch {freeze_epochs})")
    axes[0].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8,
                     label=f"Best checkpoint (epoch {best_epoch})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training / Validation Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)
 
    axes[1].plot(epochs, history["val_f1_macro"], marker="D", color="#C97B63")
    axes[1].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6)
    axes[1].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8)
    best_idx = epochs.index(best_epoch)
    axes[1].scatter([best_epoch], [history["val_f1_macro"][best_idx]], color="green", s=100, zorder=5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Val F1 (macro)")
    axes[1].set_title("Validation F1-macro")
    axes[1].grid(alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "01_training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_confusion_matrix(cm: np.ndarray, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
    im0 = axes[0].imshow(cm, cmap="Oranges")
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_yticks(range(len(class_names)))
    axes[0].set_xticklabels(class_names)
    axes[0].set_yticklabels(class_names)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].set_title("Confusion Matrix (count)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm[i, j]
            color = "white" if val > cm.max() * 0.5 else "black"
            axes[0].text(j, i, str(val), ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
 
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    im1 = axes[1].imshow(cm_norm, cmap="Oranges", vmin=0, vmax=1)
    axes[1].set_xticks(range(len(class_names)))
    axes[1].set_yticks(range(len(class_names)))
    axes[1].set_xticklabels(class_names)
    axes[1].set_yticklabels(class_names)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].set_title("Confusion Matrix (row-normalized)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i, j]
            color = "white" if val > 0.5 else "black"
            axes[1].text(j, i, f"{val:.0%}", ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "02_confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_per_class_metrics(all_labels, all_preds, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_fscore_support
 
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(len(class_names)), zero_division=0
    )
 
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(class_names))
    width = 0.25
 
    bars1 = ax.bar(x - width, precision, width, label="Precision", color="#E2B08C")
    bars2 = ax.bar(x, recall, width, label="Recall", color="#C97B63")
    bars3 = ax.bar(x + width, f1, width, label="F1-score", color="#8C9EE2")
 
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
 
    ax.set_xlabel("Class")
    ax.set_ylabel("Score")
    ax.set_title("Per-class Precision / Recall / F1 (Test Set)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n(n={s})" for c, s in zip(class_names, support)])
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "03_per_class_metrics.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
    return precision, recall, f1, support
 
 
def plot_imbalance_vs_recall(support, recall, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(10, 6))
 
    ax1.bar(class_names, support, color="#D9CFC1", alpha=0.7, label="Test set support")
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Support (# samples)", color="#8C7B6C")
    ax1.tick_params(axis="y", labelcolor="#8C7B6C")
 
    ax2 = ax1.twinx()
    ax2.plot(class_names, recall, marker="o", color="#C0392B", linewidth=2.5, markersize=8)
    ax2.set_ylabel("Recall", color="#C0392B")
    ax2.tick_params(axis="y", labelcolor="#C0392B")
    ax2.set_ylim(0, 1.0)
    for i, r in enumerate(recall):
        ax2.annotate(f"{r:.2f}", (i, r), textcoords="offset points", xytext=(0, 10),
                     ha="center", color="#C0392B", fontweight="bold")
 
    ax1.set_title("Class Imbalance vs Recall")
    fig.tight_layout()
    path = os.path.join(output_dir, "04_imbalance_vs_recall.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_yolo_style_results(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    """仿 YOLO results.png 風格：一張圖網格顯示所有指標隨 epoch 的折線走勢"""
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
    best_idx = epochs.index(best_epoch)
 
    panels = [
        ("train_loss", "Train Loss", "#E2B08C"),
        ("val_loss", "Val Loss", "#8C9EE2"),
        ("val_precision_macro", "Val Precision (macro)", "#C97B63"),
        ("val_recall_macro", "Val Recall (macro)", "#6BA383"),
        ("val_f1_macro", "Val F1 (macro)", "#C0392B"),
        ("val_accuracy", "Val Accuracy", "#8E6C88"),
    ]
 
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
 
    for ax, (key, title, color) in zip(axes, panels):
        values = history[key]
        ax.plot(epochs, values, marker="o", markersize=4, color=color, linewidth=1.8)
        # 用平滑線 (簡單移動平均) 疊加，YOLO 風格會有一條淡色 raw + 一條平滑趨勢線
        if len(values) >= 5:
            window = 3
            smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
            smooth_epochs = epochs[window - 1:]
            ax.plot(smooth_epochs, smoothed, color=color, linewidth=2.5, alpha=0.9, linestyle="--")
        ax.axvline(x=freeze_epochs, color="gray", linestyle=":", alpha=0.5)
        ax.axvline(x=best_epoch, color="green", linestyle=":", alpha=0.7)
        ax.scatter([best_epoch], [values[best_idx]], color="green", s=60, zorder=5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.grid(alpha=0.3)
 
    fig.suptitle(
        f"Training Results  (backbone unfreeze @ epoch {freeze_epochs}, best checkpoint @ epoch {best_epoch})",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    path = os.path.join(output_dir, "00_results_grid.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_summary_table(all_labels, all_preds, class_names: list, output_dir: str):
    """產出各類別 + 整體彙總的表格圖與CSV，方便直接複製進報告"""
    import matplotlib.pyplot as plt
    from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                                  precision_score, recall_score, f1_score)
 
    # --- 各類別指標 ---
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(len(class_names)), zero_division=0
    )
 
    rows = []
    for i, name in enumerate(class_names):
        rows.append([name, f"{precision[i]:.3f}", f"{recall[i]:.3f}", f"{f1[i]:.3f}", str(support[i])])
 
    # --- 整體彙總指標 (合一數據) ---
    accuracy = accuracy_score(all_labels, all_preds)
    total_support = len(all_labels)
 
    macro_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_r = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
 
    weighted_p = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    weighted_r = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
 
    rows.append(["", "", "", "", ""])  # 空行分隔各類別與整體彙總
    rows.append(["Accuracy", "", "", f"{accuracy:.3f}", str(total_support)])
    rows.append(["Macro Avg", f"{macro_p:.3f}", f"{macro_r:.3f}", f"{macro_f1:.3f}", str(total_support)])
    rows.append(["Weighted Avg", f"{weighted_p:.3f}", f"{weighted_r:.3f}", f"{weighted_f1:.3f}", str(total_support)])
 
    columns = ["Class", "Precision", "Recall", "F1-score", "Support"]
 
    # --- 存成 CSV，方便直接開Excel或複製貼上 ---
    csv_path = os.path.join(output_dir, "05_summary_table.csv")
    with open(csv_path, "w", encoding="utf-8-sig") as f:
        f.write(",".join(columns) + "\n")
        for r in rows:
            f.write(",".join(r) + "\n")
    print(f"[table] 儲存: {csv_path}")
 
    # --- 存成表格圖，方便直接放進簡報/報告 ---
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(rows) + 1.5))
    ax.axis("off")
 
    table = ax.table(cellText=rows, colLabels=columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)
 
    # 標題列樣式
    for j in range(len(columns)):
        table[0, j].set_facecolor("#4A4A4A")
        table[0, j].set_text_props(color="white", fontweight="bold")
 
    # 各類別列 (前 len(class_names) 列，index+1因為有標題列)
    for i in range(len(class_names)):
        for j in range(len(columns)):
            table[i + 1, j].set_facecolor("#F5F0EB")
 
    # 整體彙總列 (最後3列，跳過空行) 用不同底色凸顯
    summary_start = len(class_names) + 2  # +1標題列 +1空行
    for i in range(summary_start, summary_start + 3):
        for j in range(len(columns)):
            table[i, j].set_facecolor("#D9E4DD")
            table[i, j].set_text_props(fontweight="bold")
 
    ax.set_title("Classification Summary: Per-class + Overall (Test Set)",
                  fontsize=12, fontweight="bold", pad=15)
 
    plt.tight_layout()
    img_path = os.path.join(output_dir, "05_summary_table.png")
    plt.savefig(img_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {img_path}")
 
 

def generate_all_plots(history: dict, freeze_epochs: int, best_epoch: int,
                        test_metrics: dict, all_labels, all_preds,
                        class_names: list, output_dir: str):
    """訓練/評估完成後一次呼叫，產出全部圖表並存到 output_dir"""
    setup_cjk_font()
    os.makedirs(output_dir, exist_ok=True)
 
    plot_yolo_style_results(history, freeze_epochs, best_epoch, output_dir)
    plot_training_curves(history, freeze_epochs, best_epoch, output_dir)
    plot_confusion_matrix(test_metrics["cm"], class_names, output_dir)
    precision, recall, f1, support = plot_per_class_metrics(all_labels, all_preds, class_names, output_dir)
    plot_imbalance_vs_recall(support, recall, class_names, output_dir)
    plot_summary_table(all_labels, all_preds, class_names, output_dir)
    print(f"\n所有圖表已儲存至: {output_dir}")

def print_overall_summary(backbone_name: str, test_metrics: dict):
    """印出簡潔的整體結果彙總 (純文字版，方便複製貼上到報告/紀錄)"""
    accuracy = test_metrics["val_accuracy"]
    precision = test_metrics["val_precision_macro"]
    recall = test_metrics["val_recall_macro"]
    f1 = test_metrics["val_f1_macro"]

    header = f"[{backbone_name}] 整體結果"
    print("=" * 10 + " " + header)
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print("=" * (len(header) + 11))

print('S')

S


In [20]:
# ⭕ 修改後的寫法：先載入 long_df，再預覽視覺化，最後執行主訓練流程
#meta = load_ptbxl_metadata(CONFIG["META_CSV"], CONFIG["SCP_CSV"], CONFIG["SINGLE_LABEL_ONLY"])
#meta = clean_age(meta, CONFIG["AGE_CLIP_MAX"])
#meta = clean_sex(meta)
#long_df = build_image_index(meta, CONFIG["IMG_ROOT"], CONFIG["IMG_RATE"], CONFIG["MAX_IMG_PER_RECORD"])

# 1. 預覽資料增強
#sample_path = long_df["img_path"].iloc[0]
#visualize_augmentations(sample_path, CONFIG, n_examples=5, output_dir=CONFIG["OUTPUT_DIR"])

# 2. 執行模型訓練與評估
#model, test_metrics = main(CONFIG)

if __name__ == "__main__":
    main(CONFIG)

Device: cuda
[clean_age] 移除 38 筆缺少年齡的紀錄 (16272 -> 16234)
總影像數: 16105, 類別分布:
label
NORM    9012
MI      2509
STTC    2376
CD      1676
HYP      532
Name: count, dtype: int64
train/val/test = 11299/2394/2412


model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]


開始訓練 convnext_atto
  Backbone 已凍結，僅訓練分類頭
  batch 0/354 loss=1.5816
  batch 50/354 loss=1.5098
  batch 100/354 loss=1.5659
  batch 150/354 loss=1.5475
  batch 200/354 loss=1.5477
  batch 250/354 loss=1.4071
  batch 300/354 loss=1.5192
  batch 350/354 loss=1.5275
[convnext_atto] Epoch 01/30 | Loss: 1.4818/1.5844 | Acc: 0.2922/0.1834 | Prec: 0.1592 | Rec: 0.3645 | F1: 0.1917 | Time: 720.6s
  → ⭐ 已儲存最佳模型（F1: 0.1917）
  batch 0/354 loss=1.4745
  batch 50/354 loss=1.5104
  batch 100/354 loss=1.3959
  batch 150/354 loss=1.3391
  batch 200/354 loss=1.3619
  batch 250/354 loss=1.3974
  batch 300/354 loss=1.5330
  batch 350/354 loss=1.3369
[convnext_atto] Epoch 02/30 | Loss: 1.3893/1.5095 | Acc: 0.3581/0.2460 | Prec: 0.3965 | Rec: 0.4096 | F1: 0.2497 | Time: 706.4s
  → ⭐ 已儲存最佳模型（F1: 0.2497）
  batch 0/354 loss=1.4798
  batch 50/354 loss=1.5720
  batch 100/354 loss=1.3181
  batch 150/354 loss=1.3821
  batch 200/354 loss=1.4269
  batch 250/354 loss=1.4023
  batch 300/354 loss=1.3925
  batch 350/354

/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 27169 (\N{CJK UNIFIED IDEOGRAPH-6A21}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 22411 (\N{CJK UNIFIED IDEOGRAPH-578B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 40670 (\N{CJK UNIFIED IDEOGRAPH-9EDE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 38364 (\N{CJK UNIFIED IDEOGRAPH-95DC}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 27880 (\N{CJK UNIFIED IDEOGRAPH-6CE8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipykernel_22/2808040556.py:73: UserWarning: Glyph 21312 (\N{CJK UNIFIED IDEOGRAPH-5340}) missing from

[gradcam] 儲存: /kaggle/working/outputs/gradcam_CD_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_HYP_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_MI_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_NORM_0.png
[gradcam] 儲存: /kaggle/working/outputs/gradcam_STTC_0.png
